In [0]:
import  pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType,TimestampType,FloatType

catalog_name='ecommerce'

###Brands

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show()

##Remove spaces from Brand_name


In [0]:
df_silver=df_bronze.withColumn('brand_name',F.trim(F.col('brand_name')))

df_silver.show()

##Remove @ in brand_code which is extra

In [0]:

df_silver = df_silver.withColumn(
    "brand_id",
    F.regexp_replace(F.col("brand_id"), r'[^A-Za-z0-9]', '')
)

df_silver.show()

##Find Distinct Category Code

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
anomalies={
    "GROCERY":"GRCY",
    "BOOKS":"BKS",
    "TOYS":"TOY"
}

df_silver=df_silver.replace(anomalies,subset=["category_code"])

df_silver.select("category_code").distinct().show()


##Load All Cleanimg of Brand Tabke in silver layer

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")

###Category

In [0]:
df_bronze=spark.table(f"{catalog_name}.bronze.brz_category")

df_bronze.show(10)

Find Duplicates in categroy_id and Count

In [0]:
df_duplicates=df_bronze.groupBy("category_id").count().filter(F.col("count")>1)

display(df_duplicates)

Drop Duplicates Both hace same data which is incresing only Number of rows

In [0]:
df_silver=df_bronze.dropDuplicates(['category_id'])

display(df_silver)

Convert Category_code in UpperCase 

In [0]:
df_silver=df_silver.withColumn("category_id", F.upper(F.col("category_id")))

display(df_silver)

In [0]:
df_silver.write.format('delta')\
    .mode("overwrite")\
    .option("mergeSchema",True)\
    .saveAsTable(f"{catalog_name}.silver.slv_category")

###Products

Find Row and Column count

In [0]:
df_bronze=spark.read.table(f"{catalog_name}.bronze.brz_products")

row_count,column_count=df_bronze.count(),len(df_bronze.columns)

print(f"Row Count: {row_count}")
print(f"Column Count:{column_count}")

In [0]:
display(df_bronze)

Check weight_grams (contains "g")

In [0]:
df_bronze.select("weight_grams").show(5,truncate=False)

Remove "g" with blank Space " " and also convert into IntegerType

In [0]:
df_silver=df_bronze.withColumn(
    "weight_grams",
    F.regexp_replace(F.col("weight_grams"),"g"," ").cast(IntegerType())

)

df_silver.select("weight_grams").show(5,truncate=False)

Check length_cm and replace comma(,) to dot(.)

In [0]:
df_silver=df_silver.withColumn(
    "length_cm",
    F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType())
    
    )

df_silver.select("length_cm").show(3)

Convert Category_code and Brand_code in UpperCase

In [0]:
df_silver.select("category_code","brand_code").show(3)

In [0]:
df_silver=df_silver.withColumn("category_code",
        F.upper(F.col("category_code"))).withColumn(
        "brand_code",
        F.upper(F.col("brand_code"))
)
df_silver.select("category_code","brand_code").show(3)


Find Distinct in Material

In [0]:
df_silver.select("material").distinct().show()

Correct Spelling of some Category Names

In [0]:
df_silver=df_silver.withColumn(
    "material",
    F.when(F.col("material")=="Coton","Cotton")
     .when(F.col("material")=="Alumium","Aluminum")
     .when(F.col("material")=="Ruber","Rubber")
     .otherwise(F.col("material"))
)

df_silver.select("material").distinct().show()

Find Rating in Negative Which is Wrong

In [0]:
df_silver.filter(F.col('rating_count')<0).select("rating_count").show(3)

Correct Negative Rating into Positive Rating

In [0]:
df_silver=df_silver.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(),F.abs(F.col("rating_count")))
    .otherwise(F.lit(0))
)

Check Cleaned Data

In [0]:
df_silver.select("weight_grams","length_cm","category_code","brand_code","material","rating_count").show(10,truncate=False)

Check Final Cleaned data

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_products")

###Customers

In [0]:
df_bronze=spark.read.table(f"{catalog_name}.bronze.brz_customers")


row_count,column_count-df_bronze.count(),len(df_bronze.columns)


print(f"Row Count:{row_count}")
print(f"Column Count:{column_count}")

df_bronze.show(10)

Handle Null values in cutomer_id column

In [0]:
null_count=df_bronze.filter(F.col("customer_id").isNull()).count()

print(f"Null Values in Customer_id is:{null_count}")

In [0]:
# There are 300 null values in customet_id column.Display some of theses

df_bronze.filter(F.col("customer_id").isNull()).show(10)

Drop Data which is NULL in customer_id becausse there is no importance if main primary key is NULL

In [0]:
df_silver=df_bronze.dropna(subset=["customer_id"])


row_count=df_silver.count()

print(f"Rows count after Dropping Null Values:{row_count}")

Null Count in phone column

In [0]:
null_count=df_silver.filter(F.col("phone").isNull()).count()

print(f"Null Count in Phone Column:{null_count}")

See the Null values in Phone Column

In [0]:
df_silver.filter(F.col("phone").isNull()).show(10)

Fill Null Values with "Not Available" in Phone columns

In [0]:
df_silver=df_silver.fillna("Not Available",subset=["phone"])


# See The Null is available or not
null_count=df_silver.filter(F.col("phone").isNull()).count()
print(null_count)



Load the data in the silver Layer with name slv_customers

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

###Calender/Date

In [0]:
df_bronze=spark.read.table(f"{catalog_name}.bronze.brz_calendar")

row_count,column_count=df_bronze.count(),len(df_bronze.columns)


print(f"Row Count:{row_count}")
print(f"Coulmn Count:{column_count}")


df_bronze.show(5)

Show Schema of the Calendar table

In [0]:
df_bronze.printSchema()

Convert String DataType to Date DataType of date Column

In [0]:
from pyspark.sql.functions import to_date

df_silver=df_bronze.withColumn("date",to_date(df_bronze["date"],"dd-MM-yyyy"))

In [0]:
print(df_silver.printSchema)

df_bronze.show(5)

Find Duplicates in DataFrame and Remove

In [0]:
duplicates=df_silver.groupBy('date').count().filter("count>1")

# Show Duplicates

print(f"Total_duplicates:{duplicates.count()}")

display(duplicates)

In [0]:
df_silver=df_silver.dropDuplicates(['date'])

row_count=df_silver.count()

print(f"Rows after Dropping from date column:{row_count}")

day_name Normalize casing

In [0]:

# First letter should be in capital in day_name 
df_silver=df_silver.withColumn('day_name',F.initcap(F.col("day_name")))

df_silver.show(5)

Convert Negative week_of_year to Positive

In [0]:
df_silver=df_silver.withColumn("week_of_year",F.abs(F.col("week_of_year")))
df_silver.show(3)

Enhance quater and week_of_year Column

In [0]:
from pyspark.sql.functions import concat_ws, lit, col

df_silver = df_silver.withColumn(
    "quarter",
    F.concat_ws("",F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year")))
    
)

df_silver = df_silver.withColumn(
    "week_of_year",
    F.concat_ws("",F.concat(F.lit("Week"), F.col("week_of_year"), F.lit("-"), F.col("year")))
    
)

df_silver.show(3)


Rename a column week_of_year to week

In [0]:
df_silver=df_silver.withColumnRenamed("week_of_year","week")

In [0]:
df_silver.show(5)

In [0]:
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("mergeSchema", "true")\
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")